# Clustering Whole Model

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택'

In [1]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## Import Module

In [3]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [4]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


## Load CNN Model for MNIST (CNN)

In [5]:
model = tf.keras.models.load_model('/content/drive/MyDrive/files/save/baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

In [6]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9900000095367432


## Baseline model weight 확인 : 추후 clustered model의 weight와 비교 예정

In [7]:
print(model.layers[1].get_weights()[0])

[[[[ 1.57714173e-01 -5.83508331e-03 -1.61653664e-02  2.49363318e-01
     6.52938113e-02  1.06829591e-01 -1.71385765e-01 -2.76716232e-01
    -1.74441591e-01  4.45471644e-01  9.45033357e-02 -2.92737842e-01
     2.09057033e-01 -2.12433049e-03  4.11112234e-02  1.33311868e-01
     1.77187771e-02  9.29089114e-02  4.53191157e-03 -1.65191703e-02
     3.32915604e-01 -4.92629588e-01  2.01682627e-01 -2.65789870e-02
    -3.61835420e-01 -5.44804484e-02 -1.44337222e-01 -3.30566794e-01
    -8.56284052e-03  2.09492385e-01  4.19246167e-01 -3.33595902e-01]]

  [[ 1.36669710e-01 -2.66587228e-01  9.36541408e-02  2.13598669e-01
    -5.14182774e-03  7.96611980e-02  5.86670032e-03  4.99681458e-02
     2.09513828e-02  2.33950630e-01 -2.24051282e-01  6.60920888e-02
     1.57601282e-01 -2.29333758e-01  1.51049420e-01  1.55506238e-01
     1.50379196e-01  3.97728849e-03  1.59188360e-01 -1.00994974e-01
    -4.84448019e-03 -7.79996216e-02 -1.22482933e-01  3.47033776e-02
    -4.30598319e-01  1.55301467e-01 -4.674925

## Clustering Whole Model
* number of cluserts : 16
* cluster centroids init : Linear ( Linear or KMean++ 추천 )

In [8]:
clustering_params = {
  'number_of_clusters': 16,
  'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.LINEAR
}

clustered_model = tfmot.clustering.keras.cluster_weights(model,**clustering_params)

## Compile 후 model 확인 : clustering을 위해 metadata가 추가 된 model 확인

In [9]:
clustered_model.compile(loss=keras.losses.SparseCategoricalCrossentropy(),
                        optimizer=keras.optimizers.Adam(learning_rate = 1e-5),  # 작은 leraning rate 사용
                        metrics=['accuracy'])

clustered_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 cluster_reshape (ClusterWe  (None, 28, 28, 1)         0         
 ights)                                                          
                                                                 
 cluster_conv2d (ClusterWei  (None, 26, 26, 32)        624       
 ghts)                                                           
                                                                 
 cluster_max_pooling2d (Clu  (None, 13, 13, 32)        0         
 sterWeights)                                                    
                                                                 
 cluster_conv2d_1 (ClusterW  (None, 11, 11, 16)        9248      
 eights)                                                         
                                                                 
 cluster_max_pooling2d_1 (C  (None, 5, 5, 16)          0

## Clustering을 위한 training

In [10]:
hist_clustered = clustered_model.fit(
    train_images,
    train_labels,
    batch_size=500,
    epochs=1,
    validation_split=0.1
)

108/108 [==============================] - 33s 284ms/step - loss: 0.0091 - accuracy: 0.9970 - val_loss: 0.0416 - val_accuracy: 0.9905


## Accuracy 비교 : baseline model vs clustered model

In [11]:
_, clustered_model_accuracy = clustered_model.evaluate(
  test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Clustered test accuracy:', clustered_model_accuracy)

Baseline test accuracy: 0.9900000095367432
Clustered test accuracy: 0.9890000224113464


## Clustered model의 weight 확인 : 16개의 cluster로 weight가 제한 됨

In [12]:
final_model = tfmot.clustering.keras.strip_clustering(clustered_model)

print(final_model.layers[1].get_weights()[0])

[[[[ 0.17803034  0.02604295 -0.04840112  0.253436    0.10226125
     0.10226125 -0.20152217 -0.27727422 -0.20152217  0.48234993
     0.10226125 -0.27727422  0.17803034  0.02604295  0.02604295
     0.10226125  0.02604295  0.10226125  0.02604295 -0.04840112
     0.33015049 -0.50435275  0.17803034 -0.04840112 -0.35246617
    -0.04840112 -0.12557879 -0.35246617  0.02604295  0.17803034
     0.4050217  -0.35246617]]

  [[ 0.10226125 -0.27727422  0.10226125  0.17803034  0.02604295
     0.10226125  0.02604295  0.02604295  0.02604295  0.253436
    -0.20152217  0.10226125  0.17803034 -0.20152217  0.17803034
     0.17803034  0.17803034  0.02604295  0.17803034 -0.12557879
     0.02604295 -0.04840112 -0.12557879  0.02604295 -0.42899597
     0.17803034 -0.50435275  0.253436    0.17803034  0.17803034
     0.33015049 -0.12557879]]

  [[ 0.02604295 -0.20152217  0.17803034  0.33015049 -0.12557879
    -0.27727422  0.02604295  0.10226125 -0.04840112  0.253436
    -0.35246617  0.10226125 -0.04840112 -0.125

In [13]:
len(set(list(final_model.layers[1].get_weights()[0].reshape((-1)))))

16

## 최종 clustered model의 구조 확인
* baseline model과 같음 : memory size에서도 달라진 부분이 없음 (TFMOT clustering 의 한계)

In [14]:
final_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

## TFMOT에 의해 만들어진 clustered model의 의미

---


* huffman coding에 의해 압축할 때 효율적인 압축이 가능한 형태로 변경

In [15]:
import tempfile
import os
import zipfile

def get_zipped_model_size(model):

  _, model_file = tempfile.mkstemp('.h5')
  model.save(model_file)

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(model_file)

  return os.path.getsize(zipped_file)

## .zip 로 압축된 파일 크기 비교 : baseline model vs clustered model

In [16]:
size_zipped_baseline_model = get_zipped_model_size(model)
size_zipped_clustered_model = get_zipped_model_size(final_model)

print("Size of zipped baseline model file : {}".format(size_zipped_baseline_model))
print("Size of zipped clustered model file : {}".format(size_zipped_clustered_model))
print("ratio : {}".format(size_zipped_baseline_model/size_zipped_clustered_model))

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Size of zipped baseline model file : 442274
Size of zipped clustered model file : 34891
ratio : 12.675876300478633
